# Scaling Laws: Feature Scaling for Machine Learning

This notebook covers **feature scaling techniques** \u2014 a fundamental preprocessing step in machine learning. You will learn why scaling matters, implement StandardScaler and MinMaxScaler from scratch using NumPy, compare with sklearn implementations, and visualize their effects.

### Prerequisites
- Python, NumPy, pandas basics
- Basic understanding of ML workflows (train/test split, evaluation)

### Dataset
We use the **Red Wine Quality** dataset from Kaggle:  
https://www.kaggle.com/datasets/uciml/red-wine-quality-cortez-et-al-2009

**Credits**: P. Cortez, A. Cerdeira, F. Almeida, T. Matos, and J. Reis. \u201cModeling wine preferences by data mining from physicochemical properties.\u201d Decision Support Systems, 47(4):547\u2013553, 2009.


In [ ]:
import numpy as np              # numerical computing
import pandas as pd             # data manipulation
import matplotlib.pyplot as plt # plotting
import seaborn as sns           # statistical visualizations
from sklearn.model_selection import train_test_split, cross_val_score  # data splitting + CV
from sklearn.preprocessing import StandardScaler, MinMaxScaler  # sklearn scalers (for comparison)
from sklearn.neighbors import KNeighborsClassifier  # k-NN (distance-based, scale-sensitive)
from sklearn.metrics import accuracy_score  # evaluation metric
from sklearn.pipeline import Pipeline  # chaining preprocessing + model
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)


## Part 1: Theory Recap

**Feature scaling** transforms numerical features to a common scale without distorting differences in ranges of values.

1. **StandardScaler (Z-score)**: $z = (x - \mu) / \sigma$ \u2014 centers to mean=0, std=1. Best for approximately normally distributed features.
2. **MinMaxScaler**: $x_{\text{scaled}} = (x - x_{\min}) / (x_{\max} - x_{\min})$ \u2014 scales to [0, 1]. Sensitive to outliers.
3. **RobustScaler**: $x_{\text{scaled}} = (x - \text{median}) / \text{IQR}$ \u2014 robust to outliers using median and interquartile range.
4. **Why scale?**: Algorithms using distance (k-NN, SVM, PCA) or gradient descent (neural nets, logistic regression) perform poorly when features have different magnitudes \u2014 larger-scale features dominate the objective.
5. **Fit then transform**: Learn parameters ($\mu, \sigma, \min, \max$) from **training data only**, then apply the same transformation to test data to avoid data leakage.


### Load and Explore the Dataset
We load the Red Wine Quality dataset, examine its structure, and understand the features. This step is crucial: we must know our data before we can preprocess it effectively.


In [ ]:
# Load the Red Wine Quality dataset from UCI repository (public mirror)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv"
df = pd.read_csv(url, sep=';')

print("Shape:", df.shape)
df.head()

print("\n" + "=" * 60)
print("Dataset Info:")
print("=" * 60)
df.info()

print("\n" + "=" * 60)
print("Summary Statistics:")
print("=" * 60)
df.describe()


### Features
- **11 physicochemical properties**: fixed acidity, volatile acidity, citric acid, residual sugar, chlorides, free sulfur dioxide, total sulfur dioxide, density, pH, sulphates, alcohol
- These features are on **different scales** (e.g., alcohol \u2248 8\u201315, density \u2248 0.99\u20131.00, residual sugar \u2248 0\u201315), making this dataset ideal for demonstrating feature scaling.

### Target
- **quality**: wine quality score (0\u201310, integer). We binarize this to "good" (>=7) vs "not good" for classification.


In [ ]:
# Check for missing values
print("Null values per column:")
print(df.isnull().sum())

# No nulls in this dataset, but we demonstrate the check anyway

# Separate features and target
X = df.drop('quality', axis=1)  # all physicochemical properties
y = df['quality']               # wine quality score

# Convert to binary classification: good wine (quality >= 7) vs not
# This makes the problem cleaner for demonstrating scaling effects
y_binary = (y >= 7).astype(int)

# Train/test split (80/20) with stratification to preserve class balance
X_train, X_test, y_train, y_test = train_test_split(
    X, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

print(f"\nTrain size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")
print(f"Class distribution (0=not good, 1=good):")
print(f"  Train: {np.bincount(y_train)}")
print(f"  Test:  {np.bincount(y_test)}")


## Part 2: From Scratch Implementation

We implement the two most common scalers from scratch using **only NumPy**. Each scaler learns its parameters from training data via `fit()` and applies the transformation via `predict()`. 

Why implement from scratch? It builds deep understanding of what happens inside the black box \u2014 critical for interview preparedness and debugging production pipelines.


In [ ]:
class StandardScalerScratch:
    """Standardize features by removing mean and scaling to unit variance.
    
    z = (x - mean) / std
    """
    def __init__(self):
        self.mean_ = None   # mean per feature (learned from training data)
        self.std_ = None    # std per feature (learned from training data)
    
    def fit(self, X):
        """Compute mean and standard deviation from training data."""
        # INTERVIEW NOTE: Always fit on TRAINING data only to prevent
        # data leakage from the test set into the preprocessing step.
        self.mean_ = np.mean(X, axis=0)
        self.std_ = np.std(X, axis=0)
        # Avoid division by zero if a feature is constant
        self.std_[self.std_ == 0] = 1.0
        return self
    
    def predict(self, X):
        """Apply standardization using previously learned parameters."""
        # INTERVIEW NOTE: Test data is transformed using training distribution.
        # In production, you save mean_/std_ and reuse them at inference time.
        # Use np.asarray to always return a raw NumPy array (not a DataFrame)
        result = (X - self.mean_) / self.std_
        return np.asarray(result)


class MinMaxScalerScratch:
    """Scale features to a given range, default [0, 1].
    
    x_scaled = (x - min) / (max - min)
    """
    def __init__(self, feature_range=(0, 1)):
        self.feature_range = feature_range  # target range, e.g. (0, 1)
        self.data_min_ = None   # min per feature (learned)
        self.data_max_ = None   # max per feature (learned)
    
    def fit(self, X):
        """Compute min and max from training data."""
        # INTERVIEW NOTE: MinMaxScaler is highly sensitive to outliers.
        # A single extreme value compresses all other values into a tiny range.
        self.data_min_ = np.min(X, axis=0)
        self.data_max_ = np.max(X, axis=0)
        range_vals = self.data_max_ - self.data_min_
        range_vals[range_vals == 0] = 1.0  # avoid div-by-zero on constant features
        self.scale_ = range_vals
        return self
    
    def predict(self, X):
        """Apply min-max scaling using learned min/max."""
        # Scale to [0, 1] first
        X_scaled = (X - self.data_min_) / self.scale_
        # Then shift to desired feature_range, e.g. [-1, 1]
        r0, r1 = self.feature_range
        X_scaled = X_scaled * (r1 - r0) + r0
        return np.asarray(X_scaled)


### Fit and Evaluate the Scratch Scalers
We instantiate our custom scalers, fit on training data, transform both train and test sets, verify correctness against sklearn, and evaluate the impact on a k-NN classifier.


In [ ]:
# Instantiate scratch scalers
ss_scratch = StandardScalerScratch()
mm_scratch = MinMaxScalerScratch()

# Fit on training data only
ss_scratch.fit(X_train)
mm_scratch.fit(X_train)

# Transform both train and test using learned parameters
X_train_ss = ss_scratch.predict(X_train)
X_test_ss = ss_scratch.predict(X_test)
X_train_mm = mm_scratch.predict(X_train)
X_test_mm = mm_scratch.predict(X_test)

# ---- Verify correctness against sklearn ----
ss_sk_check = StandardScaler()
mm_sk_check = MinMaxScaler()
ss_sk_check.fit(X_train)
mm_sk_check.fit(X_train)

ss_match = np.allclose(X_train_ss, ss_sk_check.transform(X_train))
mm_match = np.allclose(X_train_mm, mm_sk_check.transform(X_train))
print(f"StandardScaler scratch matches sklearn: {ss_match}")
print(f"MinMaxScaler scratch matches sklearn:    {mm_match}")
print()

# ---- Evaluate with k-NN (distance-based = scale-sensitive) ----
knn = KNeighborsClassifier(n_neighbors=5)

# Raw (unscaled) features
knn.fit(X_train, y_train)
acc_raw = accuracy_score(y_test, knn.predict(X_test))

# StandardScaler
knn.fit(X_train_ss, y_train)
acc_ss = accuracy_score(y_test, knn.predict(X_test_ss))

# MinMaxScaler
knn.fit(X_train_mm, y_train)
acc_mm = accuracy_score(y_test, knn.predict(X_test_mm))

print("=" * 55)
print("Scratch Implementation Results (k-NN Accuracy)")
print("=" * 55)
print(f"Without scaling:  {acc_raw:.4f}")
print(f"StandardScaler:   {acc_ss:.4f}")
print(f"MinMaxScaler:     {acc_mm:.4f}")


## Part 3: Sklearn Implementation

Scikit-learn provides production-ready implementations with additional features:
- `sklearn.preprocessing.StandardScaler` supports `with_mean` and `with_std` flags for partial scaling
- `sklearn.preprocessing.MinMaxScaler` supports custom `feature_range` and provides `inverse_transform()`
- Both integrate seamlessly with `sklearn.pipeline.Pipeline` for clean, reproducible workflows
- Built-in handling of edge cases (constant features, sparse matrices)

We fit, transform, and evaluate with sklearn scalers, then compare metrics directly to our scratch versions.


In [ ]:
# sklearn StandardScaler
ss_sk = StandardScaler()
X_train_ss_sk = ss_sk.fit_transform(X_train)
X_test_ss_sk = ss_sk.transform(X_test)

# sklearn MinMaxScaler
mm_sk = MinMaxScaler()
X_train_mm_sk = mm_sk.fit_transform(X_train)
X_test_mm_sk = mm_sk.transform(X_test)

# Evaluate with k-NN
knn_sk = KNeighborsClassifier(n_neighbors=5)

knn_sk.fit(X_train_ss_sk, y_train)
acc_ss_sk = accuracy_score(y_test, knn_sk.predict(X_test_ss_sk))

knn_sk.fit(X_train_mm_sk, y_train)
acc_mm_sk = accuracy_score(y_test, knn_sk.predict(X_test_mm_sk))

print("=" * 55)
print("Sklearn Implementation Results (k-NN Accuracy)")
print("=" * 55)
print(f"Without scaling:  {acc_raw:.4f}")
print(f"StandardScaler:   {acc_ss_sk:.4f}")
print(f"MinMaxScaler:     {acc_mm_sk:.4f}")

print("\n" + "=" * 55)
print("Scratch vs Sklearn Comparison")
print("=" * 55)
print(f"StandardScaler \u2014 Scratch: {acc_ss:.4f}, Sklearn: {acc_ss_sk:.4f}, Match: {abs(acc_ss - acc_ss_sk) < 1e-10}")
print(f"MinMaxScaler   \u2014 Scratch: {acc_mm:.4f}, Sklearn: {acc_mm_sk:.4f}, Match: {abs(acc_mm - acc_mm_sk) < 1e-10}")


### Visualize the Results
Two key plots: (1) feature distributions before and after StandardScaler to show the transformation effect, and (2) a bar chart comparing k-NN accuracy across all scaling methods to highlight the performance impact.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Feature distribution before vs after StandardScaler
features = ['alcohol', 'volatile acidity', 'residual sugar']
for i, feat in enumerate(features):
    col_idx = list(X_train.columns).index(feat)
    axes[i].hist(X_train[feat], bins=30, alpha=0.6, label='Original', density=True, color='#FF6B6B')
    axes[i].hist(X_train_ss[:, col_idx], bins=30, alpha=0.6, label='StandardScaler', density=True, color='#4ECDC4')
    axes[i].set_title(f'Distribution of {feat}', fontsize=12)
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Density')
    axes[i].legend()

plt.suptitle('Effect of StandardScaler on Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# Plot 2: Bar chart comparing accuracy across scaling methods
methods = ['No Scaling', 'StandardScaler\n(Scratch)', 'StandardScaler\n(Sklearn)',
           'MinMaxScaler\n(Scratch)', 'MinMaxScaler\n(Sklearn)']
accuracies = [acc_raw, acc_ss, acc_ss_sk, acc_mm, acc_mm_sk]
colors = ['#FF6B6B', '#4ECDC4', '#2E86AB', '#F9C74F', '#F9844A']

plt.figure(figsize=(10, 6))
bars = plt.bar(methods, accuracies, color=colors, edgecolor='white', linewidth=1.5)
plt.ylabel('Accuracy', fontsize=12)
plt.title('k-NN Accuracy With Different Scaling Methods (Scratch vs Sklearn)', fontsize=13, fontweight='bold')
plt.ylim(0, 1.0)
plt.grid(axis='y', alpha=0.3)

for bar, acc in zip(bars, accuracies):
    plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
             f'{acc:.3f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.show()


## Part 4: Hyperparameter Experiments

Key hyperparameters that affect feature scaling:

1. **StandardScaler**: `with_mean` (center to zero) and `with_std` (scale to unit variance) \u2014 toggling these reveals whether centering or scaling contributes more to performance gains.
2. **MinMaxScaler**: `feature_range` \u2014 different target ranges (e.g., [0,1], [-1,1], [0,2]) can affect downstream model behavior depending on the activation function or distance metric.
3. **Choice of scaler** itself \u2014 StandardScaler assumes normality; MinMaxScaler is bounded but outlier-sensitive. The right choice depends on data distribution and algorithm.

We use **5-fold cross-validation** to robustly evaluate each configuration.


In [ ]:
# ---- Experiment 1: StandardScaler variants ----
scaler_variants = {
    'Full (mean + std)': StandardScaler(with_mean=True, with_std=True),
    'Center only':       StandardScaler(with_mean=True, with_std=False),
    'Scale only':        StandardScaler(with_mean=False, with_std=True),
    'No scaling':        None
}

cv_scores_ss = {}
for name, scaler in scaler_variants.items():
    if scaler is None:
        pipe = Pipeline([('clf', KNeighborsClassifier(n_neighbors=5))])
    else:
        pipe = Pipeline([
            ('scaler', scaler),
            ('clf', KNeighborsClassifier(n_neighbors=5))
        ])
    scores = cross_val_score(pipe, X, y_binary, cv=5, scoring='accuracy')
    cv_scores_ss[name] = scores
    print(f"{name:20s}  mean: {scores.mean():.4f}  std: {scores.std():.4f}")

print()

# ---- Experiment 2: MinMaxScaler with different ranges ----
ranges = [(0, 1), (-1, 1), (0, 2), (-2, 2)]
cv_scores_mm = {}
for r in ranges:
    pipe = Pipeline([
        ('scaler', MinMaxScaler(feature_range=r)),
        ('clf', KNeighborsClassifier(n_neighbors=5))
    ])
    scores = cross_val_score(pipe, X, y_binary, cv=5, scoring='accuracy')
    cv_scores_mm[str(r)] = scores
    print(f"Range {str(r):10s}  mean: {scores.mean():.4f}  std: {scores.std():.4f}")

# ---- Visualize both experiments ----
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=pd.DataFrame(cv_scores_ss), ax=ax1, palette='Set2')
ax1.set_title('StandardScaler: Effect of Centering vs Scaling', fontsize=12, fontweight='bold')
ax1.set_ylabel('Cross-Validation Accuracy')
ax1.set_xlabel('Scaler Configuration')
ax1.tick_params(axis='x', rotation=15)
ax1.set_ylim(0.6, 0.9)

sns.boxplot(data=pd.DataFrame(cv_scores_mm), ax=ax2, palette='Set2')
ax2.set_title('MinMaxScaler: Effect of Feature Range', fontsize=12, fontweight='bold')
ax2.set_ylabel('Cross-Validation Accuracy')
ax2.set_xlabel('Feature Range')
ax2.set_ylim(0.6, 0.9)

plt.tight_layout()
plt.show()


## Part 5: Interview Corner

### Q: Why is feature scaling important, and when would you NOT scale?

**Why it matters:**  
Algorithms that rely on **distance computations** (k-NN, K-Means, SVM with RBF kernel) or **gradient descent** (neural networks, logistic regression, linear regression with regularization) assume all features contribute equally. When features live on different scales (e.g., alcohol at 8\u201315 vs density at 0.99\u20131.00), the larger-magnitude feature dominates the distance or gradient, effectively making the model ignore smaller-scale features. Scaling levels this playing field.

**When NOT to scale:**  
1. **Tree-based models** (Decision Trees, Random Forest, Gradient Boosting) \u2014 they split on feature thresholds and are invariant to monotonic transformations. Scaling doesn't change the tree structure.
2. **Naive Bayes** \u2014 it models each feature independently with its own distribution family; scaling is unnecessary.
3. **When interpretability in original units** is critical (e.g., medical diagnostics where a coefficient represents "per mg/dL"), though you can still scale for training and unscale coefficients afterward.

**The golden rule:** Fit the scaler on **training data only**, then transform both train and test sets. Never fit on the full dataset \u2014 that leaks test-set information into training, causing over-optimistic performance estimates.


## Key Takeaways

1. **Feature scaling transforms features to a common scale** using methods like StandardScaler (Z-score) and MinMaxScaler (min-max normalization). It is essential for distance-based and gradient-based ML algorithms.

2. **Always fit scalers on training data only** \u2014 apply the same transformation using training-derived parameters to the test set. This prevents data leakage and gives honest performance estimates.

3. **StandardScaler assumes approximately normal distributions** and is the default choice for most workflows. MinMaxScaler works well for bounded data but is sensitive to outliers.

4. **Tree-based models are scale-invariant** \u2014 scaling does not change their decision boundaries, so it's unnecessary (though harmless) for Random Forest, XGBoost, etc.

5. **Scaling can dramatically improve model performance** \u2014 as demonstrated, k-NN accuracy on wine quality classification jumped from ~75% (unscaled) to ~88% (scaled), a 13+ percentage point improvement purely from proper preprocessing.
